# 04 · Semantic search service

The "build search over our catalogue" round. You will embed documents with a small local model (cached, so you never pay twice), index them in FAISS, and write a `search()` that has top-k, metadata filters and an optional hybrid BM25 mode. Then you *measure* it: recall@k on labelled queries, and p50/p95 latency. Finally you expose it through FastAPI with tests, and see what the ANN knobs trade away at scale.

**Time:** ~35 min · **Runs:** offline on CPU in about a minute (loading sentence-transformers is the slow part) · **Needs:** [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb). [FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) goes deeper.

## Why this matters in interviews

- Semantic search is the retrieval half of every RAG system, and a product feature in its own right (`sd07`). It is a natural 45-minute build.
- The interviewer wants to see you **measure**. "It returns sensible results" is not an answer. "Recall@5 on our labelled queries is X for dense and Y for hybrid, and p95 latency is Z ms" is.
- The scaling discussion (ANN parameters, filtering, sharding, caching, re-indexing) is a system-design round in miniature.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `sd07` | Design a semantic search feature for an e-commerce catalogue. |
| `ve03` | Your vector database has 10 million documents. How does it find the nearest without comparing all 10 million? |
| `ve07` | What is the recall/latency trade-off in ANN, and how do you tune it? |
| `ve16` | How would you handle filtered vector search efficiently? |
| `ve29` | What is recall@k, MRR and nDCG — and when do you use each? |
| `ve13` | Lexical versus dense retrieval — what is the honest comparison? |
| `ve22` | How would you shard a vector index across machines? |
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |
| `ve23` | What is normalisation and when must you do it? |

## The question, as the interviewer asks it

> *"We sell about 40 products (pretend it's 4 million). Customers type things like 'something to keep my coffee hot on the train'. Build a search service: embed the catalogue, return the top k with optional category and price filters, and tell me how good and how fast it is. Then tell me what changes at 4 million."*

## The first 60 seconds

| Clarify | Why |
|---|---|
| What does a query look like: natural language, or SKUs and brand names? | Natural language favours dense embeddings. Codes and names need BM25, and usually both appear, hence **hybrid**. |
| Which filters, and are they hard constraints? | "In stock, under $50" must *never* be violated, so filter inside the search, not afterwards. |
| Latency budget and QPS? | Decides exact versus ANN search, and whether query embeddings are cached. |
| How often does the catalogue change? | Upserts and deletes versus nightly rebuilds, and a blue/green index swap when the embedding model changes. |
| Do we have click logs or labelled queries? | Without a labelled set you can't claim anything about quality. We'll make a small one. |

**The plan:** "Embed title plus description with `all-MiniLM-L6-v2`, normalised so the inner product is cosine similarity, and cache the embeddings by content hash. Use an exact FAISS inner-product index at this size. `search(q, k, category, max_price, mode)` pre-filters with an ID selector, and hybrid mode fuses the dense results with BM25 via reciprocal rank fusion. Evaluate recall@k and MRR on ten labelled queries per mode, time it for p50 and p95, and wrap it in FastAPI. Then talk about HNSW parameters and sharding for the 4-million case."

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import embed as api_embed

## Packages

```
pip install sentence-transformers faiss-cpu rank_bm25 numpy fastapi
```

| Package | What it gives you | Swap it for |
|---|---|---|
| `sentence-transformers` | `SentenceTransformer("all-MiniLM-L6-v2").encode(...)`: 384-d embeddings, local and free, a few ms per short query on a laptop CPU (well under 1 ms per text when batched) | an embeddings API (`text-embedding-3-small`, 1536-d, through `llm_setup.embed`), `bge-small`/`e5` models, or a fine-tuned domain model |
| `faiss-cpu` | exact (`IndexFlatIP`) and approximate (`IndexHNSWFlat`, `IndexIVFFlat`, PQ) nearest-neighbour search, plus search-time ID filters | a vector DB: Qdrant, Weaviate, Milvus, pgvector, OpenSearch k-NN. Same ideas, plus persistence, replication and filtering |
| `rank_bm25` | BM25 keyword scoring, for the lexical lane | OpenSearch/Elasticsearch BM25, SPLADE sparse vectors |
| `numpy` | vectors, the embedding cache, the metrics | — |
| `fastapi` | the HTTP layer and validation, with `TestClient` for tests | any web framework |

## Step 1 · The catalogue

Forty products, each with an id, title, category, price and description. One item carries a model code (`KB-2041`). Somebody will search for it, and that is where pure dense search stumbles.

In [ ]:
import hashlib
import re
import tempfile
import time

import numpy as np

CATALOG = [  # (title, category, price, description)
    ("Insulated travel mug", "kitchen", 24.0, "Stainless steel, keeps coffee hot for 6 hours on your commute, leak-proof lid."),
    ("French press coffee maker", "kitchen", 32.0, "Brews rich coffee in four minutes, glass carafe, 1 litre."),
    ("Non-stick frying pan", "kitchen", 29.0, "28 cm pan with a ceramic coating so eggs slide right off."),
    ("Cast iron skillet", "kitchen", 45.0, "Pre-seasoned heavy skillet for searing steak and baking cornbread."),
    ("High-speed blender", "kitchen", 89.0, "1200 W motor crushes ice and frozen fruit for smoothies."),
    ("Stand mixer", "kitchen", 249.0, "Kneads bread dough and whips cream, five-litre bowl, for keen bakers."),
    ("Silicone baking mat set", "kitchen", 15.0, "Reusable non-stick sheets for cookies and pastry, replaces parchment."),
    ("Electric kettle", "kitchen", 35.0, "Boils water in three minutes, temperature control for green tea."),
    ("Chef's knife", "kitchen", 60.0, "20 cm forged steel blade, balanced for slicing vegetables."),
    ("Wireless noise-cancelling headphones", "electronics", 199.0, "Blocks engine noise on flights, 30-hour battery, over-ear."),
    ("True wireless earbuds", "electronics", 79.0, "Small in-ear buds with a charging case, sweat resistant for the gym."),
    ("Wireless charging pad", "electronics", 25.0, "Charge your phone without cables, just set it on the pad."),
    ("Mechanical keyboard KB-2041", "electronics", 119.0, "Model KB-2041, hot-swappable switches, RGB backlight, USB-C."),
    ("4K computer monitor", "electronics", 329.0, "27 inch IPS display for photo editing and coding."),
    ("Portable power bank", "electronics", 39.0, "20,000 mAh battery pack that recharges a phone five times."),
    ("Smart watch", "electronics", 229.0, "Tracks heart rate, sleep and steps, GPS for runs."),
    ("Bluetooth speaker", "electronics", 59.0, "Waterproof speaker for the beach or the shower, 12-hour playtime."),
    ("Webcam 1080p", "electronics", 49.0, "Full HD camera with a built-in microphone for video calls."),
    ("Two-person tent", "outdoor", 159.0, "Lightweight backpacking tent that sleeps two, sets up in five minutes."),
    ("Down sleeping bag", "outdoor", 189.0, "Rated to minus ten degrees for winter camping."),
    ("Waterproof rain jacket", "outdoor", 129.0, "Breathable shell that keeps you dry on rainy hikes, packs into its pocket."),
    ("Trail running shoes", "outdoor", 139.0, "Grippy soles for muddy trails and long-distance races."),
    ("Road running shoes", "outdoor", 119.0, "Cushioned lightweight shoes for marathon training on pavement."),
    ("Hiking backpack 40 L", "outdoor", 99.0, "Carries a weekend of gear, rain cover and hip belt included."),
    ("Camping stove", "outdoor", 49.0, "Compact gas burner that boils a pot of water in four minutes outdoors."),
    ("Headlamp", "outdoor", 29.0, "Bright LED head torch for night hikes, rechargeable."),
    ("Insulated water bottle", "outdoor", 27.0, "Keeps drinks cold for 24 hours on the trail."),
    ("Yoga mat", "fitness", 35.0, "6 mm cushioned mat with a non-slip surface for yoga and pilates."),
    ("Adjustable dumbbells", "fitness", 299.0, "Dial the weight from 2 to 24 kg, replaces a full rack at home."),
    ("Resistance bands", "fitness", 19.0, "Five elastic bands for strength training anywhere."),
    ("Foam roller", "fitness", 25.0, "Relieves sore muscles after a workout."),
    ("Jump rope", "fitness", 12.0, "Speed rope with ball bearings for cardio and boxing training."),
    ("Learning Python", "books", 44.0, "A beginner's guide to programming in Python, with exercises."),
    ("Designing Data-Intensive Applications", "books", 49.0, "How databases, streams and distributed systems really work."),
    ("The Pragmatic Programmer", "books", 39.0, "Timeless advice on writing good software as a career developer."),
    ("Salt Fat Acid Heat", "books", 35.0, "A cookbook that teaches the four elements of good cooking."),
    ("The Hobbit", "books", 12.0, "A fantasy adventure about a hobbit, a dragon and a lot of treasure."),
    ("Atlas of the World", "books", 55.0, "Large-format maps of every country, for curious travellers."),
    ("Desk lamp", "home", 42.0, "Dimmable LED lamp with a warm light setting for late-night reading."),
    ("Ergonomic office chair", "home", 279.0, "Lumbar support and adjustable arms for long days at the desk."),
]
DOCS = [{"id": i, "title": t, "category": c, "price": p, "text": f"{t}. {d}"} for i, (t, c, p, d) in enumerate(CATALOG)]
CATEGORY = np.array([d["category"] for d in DOCS])
PRICE = np.array([d["price"] for d in DOCS])
print(f"{len(DOCS)} products; categories: { {c: int((CATEGORY == c).sum()) for c in sorted(set(CATEGORY))} }")

## Step 2 · Embed, normalise, and cache

`normalize_embeddings=True` makes every vector unit-length, so the **inner product equals cosine similarity** and `IndexFlatIP` ranks by cosine. Forgetting to normalise is a classic bug: long documents then win just because their vectors are longer (`ve23`).

The cache key is a hash of *model name + text*. Unchanged documents are never re-embedded, and changing the model automatically invalidates every entry.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
try:
    embedder = SentenceTransformer(EMBEDDER_NAME, local_files_only=True)   # cached copy, no network
except Exception:
    embedder = SentenceTransformer(EMBEDDER_NAME)                          # first run: ~90 MB download

CACHE_DIR = Path(tempfile.mkdtemp(prefix="emb_cache_"))

def cache_key(text):
    return hashlib.sha256(f"{EMBEDDER_NAME}\n{text}".encode()).hexdigest()[:24]

def embed_cached(texts):
    """Embeds only texts not seen before; returns an (n, 384) float32 matrix of unit vectors."""
    paths = [CACHE_DIR / f"{cache_key(t)}.npy" for t in texts]
    missing = [i for i, p in enumerate(paths) if not p.exists()]
    if missing:
        vecs = embedder.encode([texts[i] for i in missing], normalize_embeddings=True, batch_size=64)
        for i, v in zip(missing, vecs):
            np.save(paths[i], v.astype(np.float32))
    embed_cached.last_misses = len(missing)
    return np.stack([np.load(p) for p in paths])

t0 = time.perf_counter(); X = embed_cached([d["text"] for d in DOCS]); cold = time.perf_counter() - t0
t0 = time.perf_counter(); X2 = embed_cached([d["text"] for d in DOCS]); warm = time.perf_counter() - t0
print(f"shape {X.shape}, norms {np.linalg.norm(X, axis=1).min():.3f}..{np.linalg.norm(X, axis=1).max():.3f}")
print(f"cold: {cold * 1000:.0f} ms ({len(DOCS)} embedded)   warm: {warm * 1000:.0f} ms ({embed_cached.last_misses} embedded)")
assert embed_cached.last_misses == 0 and np.allclose(X, X2)

**The API route.** For hosted embeddings, `llm_setup.embed(texts)` calls `client.embeddings.create(model="text-embedding-3-small", ...)` when `OPENAI_API_KEY` is set. It works through the offline stand-in otherwise. The rest of the service is unchanged, with one exception: **the index has to be rebuilt**, because the vectors live in a different space with a different dimension (`rg14`).

In [ ]:
v = np.array(api_embed(["Insulated travel mug"]))
print(f"{PROVIDER} embeddings via llm_setup: shape {v.shape}, model {EMBED_MODEL}  vs local MiniLM: {X.shape[1]}-d")

## Step 3 · The index and `search()`

At 40 (or 400,000) vectors, **exact** search is fast enough and has perfect recall, so start there. `IndexFlatIP` compares the query against every row.

**Filters are applied *inside* the search** with `faiss.IDSelectorBatch`. FAISS skips rows that are not allowed, so you always get `k` valid results when `k` exist. The alternative, "search top k and then drop what doesn't match", returns too few results or none at all when the filter is selective (`ve16`).

**Hybrid mode** runs BM25 over the same allowed rows and fuses the two rankings with reciprocal rank fusion, $\sum 1/(60 + \text{rank})$.

In [ ]:
import faiss
from rank_bm25 import BM25Okapi

index = faiss.IndexFlatIP(X.shape[1])
index.add(X)

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

bm25 = BM25Okapi([tokenize(d["text"]) for d in DOCS])

def allowed_ids(category=None, max_price=None):
    mask = np.ones(len(DOCS), bool)
    if category:
        mask &= CATEGORY == category
    if max_price is not None:
        mask &= PRICE <= max_price
    return np.flatnonzero(mask)

def dense_ranking(query, ids, depth):
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)   # queries: always fresh
    params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids.astype(np.int64)))
    scores, rows = index.search(q, min(depth, len(ids)), params=params)
    return [int(r) for r in rows[0] if r >= 0], scores[0]

def bm25_ranking(query, ids, depth):
    scores = bm25.get_scores(tokenize(query))[ids]
    order = np.argsort(-scores, kind="stable")[:depth]
    return [int(ids[i]) for i in order if scores[i] > 0]          # no keyword match -> not a candidate

def search(query, k=5, category=None, max_price=None, mode="dense", depth=50):
    ids = allowed_ids(category, max_price)
    if len(ids) == 0:
        return []
    if mode == "dense":
        ranked = dense_ranking(query, ids, depth)[0]
    elif mode == "bm25":
        ranked = bm25_ranking(query, ids, depth)
    elif mode == "hybrid":
        fused = {}
        for ranking in (dense_ranking(query, ids, depth)[0], bm25_ranking(query, ids, depth)):
            for rank, doc_id in enumerate(ranking, start=1):
                fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (60 + rank)
        ranked = sorted(fused, key=lambda d: (-fused[d], d))
    else:
        raise ValueError(f"unknown mode {mode!r}")
    return [DOCS[i] for i in ranked[:k]]

for hit in search("something to keep my coffee hot on the train", k=3):
    print(f"  {hit['title']:34} {hit['category']:10} ${hit['price']:.0f}")

In [ ]:
q = "gift for someone who loves baking"
print("no filter             :", [h["title"] for h in search(q, k=3)])
print("kitchen, under $30    :", [(h["title"], h["price"]) for h in search(q, k=3, category="kitchen", max_price=30)])
print("books only            :", [h["title"] for h in search(q, k=3, category="books")])
filtered = search(q, k=3, category="kitchen", max_price=30)
assert len(filtered) == 3 and all(h["category"] == "kitchen" and h["price"] <= 30 for h in filtered)

q = "KB-2041"
for mode in ["dense", "bm25", "hybrid"]:
    print(f"{mode:6} '{q}' ->", [h["title"] for h in search(q, k=3, mode=mode)])

The model-code query shows the honest lexical-versus-dense comparison (`ve13`). BM25 nails an exact token such as `KB-2041`. MiniLM *may* get it too here, because the code also appears in the product title, but embeddings of opaque codes are unreliable, and at millions of items near-duplicate codes blur together. Hybrid keeps the BM25 win without giving up the paraphrase wins of the dense lane.

## Step 4 · Measure quality: recall@k and MRR on labelled queries

Twelve queries, each with the product ids that a human would call relevant. Most are **paraphrases** with little word overlap ("keep my coffee hot" versus "insulated travel mug"). Two are **keyword-style** (a model code, a book title).

- **recall@k**: the fraction of the relevant items that appear in the top k. It answers "did we find it at all?"
- **MRR**: the mean of 1/rank of the first relevant hit. It answers "how high up was it?"
- **nDCG** also rewards putting the *most* relevant items first when relevance is graded. Use it when you have graded labels, for example from clicks.

In [ ]:
TITLE_TO_ID = {d["title"]: d["id"] for d in DOCS}
LABELLED = [  # (query, titles a human judged relevant)
    ("something to keep my coffee hot on the train", ["Insulated travel mug"]),
    ("headphones that block out airplane noise", ["Wireless noise-cancelling headphones"]),
    ("shoes for training for a marathon", ["Road running shoes", "Trail running shoes"]),
    ("a pan that eggs won't stick to", ["Non-stick frying pan"]),
    ("shelter for two people when camping", ["Two-person tent"]),
    ("teach myself to code", ["Learning Python", "The Pragmatic Programmer"]),
    ("charge my phone with no cable", ["Wireless charging pad"]),
    ("make fruit smoothies at home", ["High-speed blender"]),
    ("stay dry when hiking in bad weather", ["Waterproof rain jacket"]),
    ("warm enough for winter camping at night", ["Down sleeping bag"]),
    ("KB-2041", ["Mechanical keyboard KB-2041"]),
    ("the hobbit book", ["The Hobbit"]),
]
LABELLED = [(q, {TITLE_TO_ID[t] for t in titles}) for q, titles in LABELLED]

def evaluate(mode, k=5):
    recalls, rrs = [], []
    for query, relevant in LABELLED:
        got = [h["id"] for h in search(query, k=k, mode=mode)]
        recalls.append(len(relevant & set(got)) / len(relevant))
        first = next((i + 1 for i, d in enumerate(got) if d in relevant), None)
        rrs.append(1 / first if first else 0.0)
    return np.mean(recalls), np.mean(rrs), recalls

RESULTS = {}
for mode in ["bm25", "dense", "hybrid"]:
    for k in [1, 3, 5]:
        RESULTS[(mode, k)] = evaluate(mode, k)
    r5, mrr, per_query = RESULTS[(mode, 5)]
    misses = [LABELLED[i][0] for i, r in enumerate(per_query) if r < 1]
    print(f"{mode:6}  recall@1={RESULTS[(mode, 1)][0]:.2f}  recall@5={r5:.2f}  MRR={mrr:.2f}   misses@5: {misses}")
assert RESULTS[("hybrid", 5)][0] >= max(RESULTS[("dense", 5)][0], RESULTS[("bm25", 5)][0])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.6))
for mode, color in [("bm25", "#adb5bd"), ("dense", "#4c6ef5"), ("hybrid", "#2f9e44")]:
    ax.plot([1, 3, 5], [RESULTS[(mode, k)][0] for k in [1, 3, 5]], "o-", color=color, label=mode, lw=2)
ax.set_xticks([1, 3, 5]); ax.set_xlabel("k"); ax.set_ylabel("recall@k"); ax.set_ylim(0, 1.05)
ax.set_title(f"Recall@k on {len(LABELLED)} labelled queries: BM25 trails on paraphrases")
ax.legend(); plt.tight_layout(); plt.show()

Twelve queries is a demo, not an evaluation. With so few, one query moves recall by 8 points. In a real project you would collect 100–500 queries from search logs, label them (or mine clicks), and report confidence intervals (see the [eval harness](05_eval_harness_from_scratch.ipynb)). Still, the shape is typical. BM25 trails badly at k = 1 on the paraphrased queries. Dense retrieval does well here, including on `KB-2041`, because that code sits in the product title and the catalogue is tiny. With millions of items and thousands of similar codes, the exact-token queries are where dense retrieval slips. Hybrid never did worse than the better lane, which is why it is the safe default.

## Step 5 · Measure speed: p50 and p95 latency

Time the whole `search()` call, embedding included, over 120 queries. Report percentiles, not the mean. **p50** is the typical request and **p95** is the slow tail that one request in twenty sees. Tail latency is what users remember, and at high fan-out (one page making many calls) it dominates (`pd41`).

In [ ]:
queries = [q for q, _ in LABELLED] * 10
for q in queries[:5]:
    search(q, mode="hybrid")                                # warm-up (first calls are slower)
lat = {}
for mode in ["dense", "hybrid"]:
    times = []
    for q in queries[:120]:
        t0 = time.perf_counter()
        search(q, k=5, mode=mode)
        times.append((time.perf_counter() - t0) * 1000)
    lat[mode] = np.array(times)
    p50, p95, p99 = np.percentile(lat[mode], [50, 95, 99])
    print(f"{mode:6}  p50={p50:5.2f} ms   p95={p95:5.2f} ms   p99={p99:5.2f} ms   mean={lat[mode].mean():5.2f} ms")

enc, scan = [], []
for q in queries[:30]:
    t0 = time.perf_counter(); qv = embedder.encode([q], normalize_embeddings=True).astype(np.float32)
    enc.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); index.search(qv, 5); scan.append((time.perf_counter() - t0) * 1000)
print(f"\nwhere the time goes (median of 30): embedding the query {np.median(enc):.2f} ms  vs  FAISS scan {np.median(scan):.3f} ms")
assert np.median(enc) > np.median(scan)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
bins = np.linspace(0, np.percentile(np.concatenate(list(lat.values())), 99), 40)
for mode, color in [("dense", "#4c6ef5"), ("hybrid", "#2f9e44")]:
    ax.hist(lat[mode], bins=bins, alpha=0.6, color=color, label=mode)
    for pct, ls in [(50, "-"), (95, "--")]:
        ax.axvline(np.percentile(lat[mode], pct), color=color, ls=ls, lw=1.5)
ax.set_xlabel("search() latency, ms (solid = p50, dashed = p95)"); ax.set_ylabel("queries")
ax.set_title("Latency distribution: a long right tail even for identical work"); ax.legend()
plt.tight_layout(); plt.show()

Two things to say. First, **the query embedding dominates**: the FAISS scan over 40 vectors takes microseconds, the model forward pass takes milliseconds. At scale the biggest wins are caching query embeddings for popular queries, batching, and a smaller model or a GPU. Second, even identical work has a tail (garbage collection, CPU scheduling, cache misses), which is why SLOs are written on p95 or p99. Your numbers depend on the machine. Under load, or with an embeddings **API** at roughly 100–300 ms per call, the tail grows a lot.

## Step 6 · Expose it: FastAPI + `TestClient`

`Literal` and `Field` constraints make FastAPI reject bad input (`mode=fuzzy`, `k=500`) with a 422 before your code runs.

In [ ]:
from typing import Literal

from fastapi import FastAPI, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

class Hit(BaseModel):
    id: int
    title: str
    category: str
    price: float

class SearchResponse(BaseModel):
    query: str
    mode: str
    took_ms: float
    hits: list[Hit]

app = FastAPI(title="catalog-search")

@app.get("/search", response_model=SearchResponse)
def search_endpoint(q: str = Query(min_length=1, max_length=200), k: int = Query(5, ge=1, le=50),
                    category: str | None = None, max_price: float | None = Query(None, gt=0),
                    mode: Literal["dense", "bm25", "hybrid"] = "hybrid"):
    t0 = time.perf_counter()
    hits = search(q, k=k, category=category, max_price=max_price, mode=mode)
    return SearchResponse(query=q, mode=mode, took_ms=(time.perf_counter() - t0) * 1000,
                          hits=[Hit(**{f: h[f] for f in ("id", "title", "category", "price")}) for h in hits])

api = TestClient(app)
r = api.get("/search", params={"q": "stay dry when hiking in bad weather", "k": 3})
print(r.status_code, [h["title"] for h in r.json()["hits"]])

def test_top_k_and_filters():
    body = api.get("/search", params={"q": "coffee", "k": 2, "category": "kitchen", "max_price": 30}).json()
    assert len(body["hits"]) == 2 and all(h["category"] == "kitchen" and h["price"] <= 30 for h in body["hits"])

def test_bad_input_is_422():
    assert api.get("/search", params={"q": "x", "mode": "fuzzy"}).status_code == 422
    assert api.get("/search", params={"q": "x", "k": 500}).status_code == 422
    assert api.get("/search", params={"q": ""}).status_code == 422

def test_empty_filter_returns_no_hits():
    assert api.get("/search", params={"q": "tent", "category": "toys"}).json()["hits"] == []

def test_model_code_found_by_hybrid():
    assert api.get("/search", params={"q": "KB-2041", "k": 1}).json()["hits"][0]["title"].endswith("KB-2041")

for t in [test_top_k_and_filters, test_bad_input_is_422, test_empty_filter_returns_no_hits, test_model_code_found_by_hybrid]:
    t(); print("PASS", t.__name__)

## The follow-up: "what changes at 4 million?"

### Approximate nearest neighbours: what you trade

Exact search is O(n) per query. At 4M × 384 floats (about 6 GB) a brute-force scan takes tens of milliseconds per query on a single core. **HNSW** builds a navigable graph and visits only a few thousand nodes per query. The knobs are:

| Knob | What it does | Turn it up for | Cost |
|---|---|---|---|
| `M` (links per node) | graph density | recall | memory, build time |
| `efConstruction` | search width while building | graph quality | build time |
| `efSearch` | search width per query | recall | latency |
| *(IVF)* `nlist` / `nprobe` | clusters / clusters visited per query | recall (`nprobe` up) | latency |
| *(PQ)* sub-quantizers | compress vectors 8–64× | memory | recall |

Here is the trade-off measured on 10,000 synthetic 384-d vectors (clustered, like real embeddings). Recall@10 is measured against exact search, which is the ground truth, while `efSearch` sweeps.

In [ ]:
rng = np.random.default_rng(0)
centers = rng.normal(size=(200, 384)).astype(np.float32)
base = centers[rng.integers(0, 200, 10_000)] + 0.35 * rng.normal(size=(10_000, 384)).astype(np.float32)
faiss.normalize_L2(base)
queries_v = base[rng.choice(len(base), 200, replace=False)] + 0.05 * rng.normal(size=(200, 384)).astype(np.float32)
faiss.normalize_L2(queries_v)

exact = faiss.IndexFlatIP(384); exact.add(base)
t0 = time.perf_counter(); _, truth = exact.search(queries_v, 10); exact_ms = (time.perf_counter() - t0) * 1000 / len(queries_v)

hnsw = faiss.IndexHNSWFlat(384, 16, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 40
t0 = time.perf_counter(); hnsw.add(base); build_s = time.perf_counter() - t0
threads = faiss.omp_get_max_threads()
faiss.omp_set_num_threads(1)                              # fair per-query timings
sweep = []
for ef in [8, 16, 32, 64, 128, 256]:
    hnsw.hnsw.efSearch = ef
    t0 = time.perf_counter(); _, got = hnsw.search(queries_v, 10); ms = (time.perf_counter() - t0) * 1000 / len(queries_v)
    recall = np.mean([len(set(g) & set(t)) / 10 for g, t in zip(got, truth)])
    sweep.append((ef, recall, ms))
    print(f"efSearch={ef:4d}  recall@10={recall:.3f}  {ms:.3f} ms/query")
faiss.omp_set_num_threads(threads)
print(f"exact search: {exact_ms:.3f} ms/query (recall 1.0 by definition); HNSW build {build_s:.1f} s")
assert sweep[-1][1] > sweep[0][1]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot([s[2] for s in sweep], [s[1] for s in sweep], "o-", color="#4c6ef5", lw=2)
for ef, rec, ms in sweep:
    ax.annotate(f"ef={ef}", (ms, rec), textcoords="offset points", xytext=(5, -12), fontsize=8)
ax.axhline(1.0, color="grey", ls=":", lw=1)
ax.scatter([exact_ms], [1.0], color="crimson", zorder=3, label=f"exact scan ({exact_ms:.2f} ms)")
ax.set_xlabel("ms per query (1 thread)"); ax.set_ylabel("recall@10 vs exact")
ax.set_title("HNSW on 10k vectors: efSearch buys recall with latency"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

Read it as the answer to `ve07`. Recall rises steeply with `efSearch` and then flattens, while latency keeps growing. **Tune `efSearch` to the smallest value that meets your recall target (say 0.95)**, measured on your own data, as here. At only 10k vectors the exact scan is still competitive. HNSW's advantage grows with n, because exact search is linear in n and HNSW is roughly logarithmic.

### The rest of the scaling story

| Topic | What to say |
|---|---|
| **Filtering at scale** (`ve16`) | A pre-filter (ID selector, as above) is exact but slow when the filter keeps very few rows. A post-filter is fast but returns too few results. Vector DBs do filtered HNSW traversal. For very selective, common filters (tenant, language), **partition the index** by that field. |
| **Sharding** (`ve22`) | Split vectors across N machines (by hash, or by tenant). Each shard returns its top k, and a coordinator merges them (scatter-gather). Latency is set by the slowest shard, so add replicas for tail latency and throughput. |
| **Memory** | 4M × 384 × 4 bytes ≈ 6 GB raw, plus the HNSW links. Product or scalar quantisation cuts that 4–32×, at some cost in recall. Rerank the top 100 with full vectors to recover it. |
| **Caching** | Cache query embeddings (popular queries repeat), cache results for the head queries, and invalidate on catalogue updates. |
| **Re-indexing** | Upserts and deletes for catalogue changes (HNSW deletes are tombstones, so compact periodically). **A new embedding model means a full re-embed into a new index**, queried by a new query encoder, then a blue/green swap. The two vector spaces are not compatible. |
| **Monitoring** | Recall on a fixed canary query set (catches silent regressions), p95 latency, the zero-result rate, and the click-through rate of the top results. |

## The whole thing as one file: `search_service.py`

```python
"""search_service.py - semantic + hybrid catalogue search behind FastAPI.
pip install sentence-transformers faiss-cpu rank_bm25 fastapi uvicorn ; run: uvicorn search_service:app"""
import re
import time
from typing import Literal

import faiss
import numpy as np
from fastapi import FastAPI, Query
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DOCS = [  # in real life: loaded from the product database
    {"id": 0, "title": "Insulated travel mug", "category": "kitchen", "price": 24.0,
     "text": "Insulated travel mug. Keeps coffee hot for 6 hours on your commute."},
    {"id": 1, "title": "Mechanical keyboard KB-2041", "category": "electronics", "price": 119.0,
     "text": "Mechanical keyboard KB-2041. Hot-swappable switches, RGB backlight."},
    {"id": 2, "title": "Two-person tent", "category": "outdoor", "price": 159.0,
     "text": "Two-person tent. Lightweight backpacking tent that sleeps two."},
]
CATEGORY = np.array([d["category"] for d in DOCS])
PRICE = np.array([d["price"] for d in DOCS])

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X = embedder.encode([d["text"] for d in DOCS], normalize_embeddings=True).astype(np.float32)
index = faiss.IndexFlatIP(X.shape[1])            # unit vectors: inner product == cosine
index.add(X)

def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

bm25 = BM25Okapi([tokenize(d["text"]) for d in DOCS])

def allowed_ids(category=None, max_price=None):
    mask = np.ones(len(DOCS), bool)
    if category:
        mask &= CATEGORY == category
    if max_price is not None:
        mask &= PRICE <= max_price
    return np.flatnonzero(mask)

def dense_ranking(query, ids, depth):
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids.astype(np.int64)))   # filter INSIDE the search
    _, rows = index.search(q, min(depth, len(ids)), params=params)
    return [int(r) for r in rows[0] if r >= 0]

def bm25_ranking(query, ids, depth):
    scores = bm25.get_scores(tokenize(query))[ids]
    order = np.argsort(-scores, kind="stable")[:depth]
    return [int(ids[i]) for i in order if scores[i] > 0]

def search(query, k=5, category=None, max_price=None, mode="hybrid", depth=50):
    ids = allowed_ids(category, max_price)
    if len(ids) == 0:
        return []
    if mode == "dense":
        ranked = dense_ranking(query, ids, depth)
    elif mode == "bm25":
        ranked = bm25_ranking(query, ids, depth)
    else:                                         # hybrid: reciprocal rank fusion
        fused = {}
        for ranking in (dense_ranking(query, ids, depth), bm25_ranking(query, ids, depth)):
            for rank, doc_id in enumerate(ranking, start=1):
                fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (60 + rank)
        ranked = sorted(fused, key=lambda d: (-fused[d], d))
    return [DOCS[i] for i in ranked[:k]]

app = FastAPI(title="catalog-search")

@app.get("/search")
def search_endpoint(q: str = Query(min_length=1, max_length=200), k: int = Query(5, ge=1, le=50),
                    category: str | None = None, max_price: float | None = Query(None, gt=0),
                    mode: Literal["dense", "bm25", "hybrid"] = "hybrid"):
    t0 = time.perf_counter()
    hits = search(q, k=k, category=category, max_price=max_price, mode=mode)
    return {"query": q, "mode": mode, "took_ms": (time.perf_counter() - t0) * 1000,
            "hits": [{f: h[f] for f in ("id", "title", "category", "price")} for h in hits]}
```

## Try it yourself

**1.** Add **nDCG@5** with graded relevance for the marathon query: road running shoes = 2 (perfect), trail running shoes = 1 (good). Compute it for dense and hybrid.

In [ ]:
# Solution — try it yourself first, then run this
def ndcg_at_k(ranked_ids, gains, k=5):
    dcg = sum(gains.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked_ids[:k]))
    ideal = sum(g / np.log2(i + 2) for i, g in enumerate(sorted(gains.values(), reverse=True)[:k]))
    return dcg / ideal if ideal else 0.0

gains = {TITLE_TO_ID["Road running shoes"]: 2, TITLE_TO_ID["Trail running shoes"]: 1}
for mode in ["dense", "hybrid", "bm25"]:
    ranked = [h["id"] for h in search("shoes for training for a marathon", k=5, mode=mode)]
    print(f"{mode:6} nDCG@5 = {ndcg_at_k(ranked, gains):.3f}   top-3: {[DOCS[i]['title'] for i in ranked[:3]]}")

**2.** Add a product ("Pour-over coffee kettle", kitchen, $38, "Gooseneck kettle for pour-over coffee, precise pouring.") **without** re-embedding the other 40. Check that the cache embedded exactly one text and that a coffee query can now find the new product.

In [ ]:
# Solution — try it yourself first, then run this
new = ("Pour-over coffee kettle", "kitchen", 38.0, "Gooseneck kettle for pour-over coffee, precise pouring.")
DOCS.append({"id": len(DOCS), "title": new[0], "category": new[1], "price": new[2], "text": f"{new[0]}. {new[3]}"})
CATEGORY, PRICE = np.append(CATEGORY, new[1]), np.append(PRICE, new[2])
X = embed_cached([d["text"] for d in DOCS])
print("texts embedded this time:", embed_cached.last_misses)
index.add(X[-1:])                                        # upsert one vector
bm25 = BM25Okapi([tokenize(d["text"]) for d in DOCS])     # BM25 here is rebuilt (cheap at this size)
print([h["title"] for h in search("kettle for making pour over coffee", k=3, mode="hybrid")])
assert embed_cached.last_misses == 1 and index.ntotal == len(DOCS)

## Say it in an interview

**30 seconds:** "I embed title plus description with a sentence-transformer, normalise so the inner product is cosine, and cache embeddings by content hash. Exact FAISS search is fine up to a few hundred thousand vectors, and HNSW beyond that. Filters run *inside* the search, never as a post-filter. Hybrid mode fuses BM25 and dense rankings with RRF, because paraphrases need dense and codes need BM25. I measure recall@k and MRR on a labelled query set, and p50/p95 latency end to end, and I serve it behind FastAPI with validated parameters."

**Numbers:** MiniLM gives 384-d vectors, a few ms per single query on CPU and much less per text when batched. OpenAI `text-embedding-3-small` is 1536-d. 4M × 384 float32 ≈ 6 GB before quantisation. RRF uses `c = 60`. Tune `efSearch` to the smallest value that meets the recall target.

**Traps:** forgetting to normalise (cosine versus inner product); post-filtering returning too few results; comparing vectors from two different models; measuring the mean instead of p95; tuning the ANN on random vectors instead of your own data; changing the embedding model without re-embedding the corpus.

## Next

- Deeper: [FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) · [hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) · [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) · [sentence-transformers and cross-encoders](../10_huggingface/04_sentence_transformers_embeddings_and_cross_encoders.ipynb) · [latency percentiles](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [semantic caching](../16_production/03_caching_exact_and_semantic.ipynb)
- Other rounds: [RAG skeleton](01_rag_skeleton.ipynb) · [eval harness from scratch](05_eval_harness_from_scratch.ipynb)
- Theory: [ai_system_design_concepts course](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)